# Notebook 04: Generate All Paper Figures
**Run after:** Notebooks 01, 02, and 03 (all results/*.pkl files must exist)

This notebook regenerates all 6 paper figures as vector PDFs,
previews them inline, and zips them for download.

All PDFs are fully zoomable in any PDF viewer (Acrobat, Preview, Okular).
Copy them directly into your LaTeX project folder alongside the .tex file.

In [ ]:
!pip install -q causal-learn matplotlib numpy pandas scipy scikit-learn networkx

In [ ]:
import os, sys, importlib
PROJECT_ROOT = '/content/causal_bias_project'  # <-- adjust
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
os.makedirs('figures/output', exist_ok=True)
print('Existing results:', os.listdir('results/'))

## Generate all 6 figures

In [ ]:
fig_modules = [
    ('figures.fig_gt_biased',          'make_figure', 'Fig 1: Ground Truth DAG (biased)'),
    ('figures.fig_sensitivity',        'main',        'Fig 2: Sensitivity Analysis Grid'),
    ('figures.fig_directlingam_loan',  'make_figure', 'Fig 3: DirectLiNGAM Synthetic'),
    ('figures.fig_compas_dag',         'make_figure', 'Fig 4: COMPAS Domain-Knowledge DAG'),
    ('figures.fig_directlingam_compas','make_figure', 'Fig 5: DirectLiNGAM COMPAS'),
    ('figures.fig_corr_vs_causal',     'make_figure', 'Fig 6: Correlation vs Causal'),
]

errors = []
for mod_name, fn_name, description in fig_modules:
    print(f'\n── Generating {description}...')
    try:
        mod = importlib.import_module(mod_name)
        importlib.reload(mod)
        getattr(mod, fn_name)()
        print(f'   ✓ Done')
    except Exception as e:
        print(f'   ✗ ERROR: {e}')
        errors.append((description, str(e)))

print('\n' + '='*50)
print('Figures generated:')
for f in sorted(os.listdir('figures/output/')):
    size = os.path.getsize(f'figures/output/{f}')
    print(f'  {f:45s} {size:>8,} bytes')

if errors:
    print(f'\n{len(errors)} errors occurred:')
    for desc, err in errors:
        print(f'  {desc}: {err}')

## Preview all figures inline (PNG versions)

In [ ]:
from IPython.display import Image, display

figure_order = [
    ('fig_gt_biased.png',            'Figure 1: Ground Truth DAG (biased synthetic dataset)'),
    ('fig_sensitivity.png',          'Figure 2: Sensitivity Analysis Grid'),
    ('fig_directlingam_loan.png',    'Figure 3: DirectLiNGAM — Biased Synthetic Dataset'),
    ('fig_compas_dag.png',           'Figure 4: COMPAS Domain-Knowledge DAG'),
    ('fig_directlingam_compas.png',  'Figure 5: DirectLiNGAM — COMPAS Dataset'),
    ('fig_corr_vs_causal.png',       'Figure 6: Correlation-Based vs. Causal Methods'),
]

for fname, caption in figure_order:
    path = f'figures/output/{fname}'
    if os.path.exists(path):
        print(f'\n── {caption}')
        display(Image(path, width=700))
    else:
        print(f'\n── {caption}: NOT FOUND (run generation cell above)')

## Download all PDFs as a zip file

In [ ]:
import zipfile
from google.colab import files

zip_path = 'paper_figures.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir('figures/output/')):
        if fname.endswith('.pdf'):
            fpath = f'figures/output/{fname}'
            zf.write(fpath, fname)
            print(f'  Added: {fname}')

print(f'\nZip size: {os.path.getsize(zip_path):,} bytes')
print('Downloading...')
files.download(zip_path)

## How to use the PDFs in your LaTeX file

1. Extract `paper_figures.zip`
2. Copy all `.pdf` files into the same folder as `CausalityAndBias_ICMLA2026.tex`
3. The `.tex` file already references them by name:
   - `\includegraphics[width=\columnwidth]{fig_gt_biased.pdf}`
   - `\includegraphics[width=\columnwidth]{fig_sensitivity.pdf}`
   - etc.
4. Compile with `pdflatex CausalityAndBias_ICMLA2026.tex`

**Why PDF figures?**
PDF figures are vector graphics — they remain perfectly sharp at any zoom level
in Acrobat, Preview, or any PDF viewer. PNG figures will pixelate when zoomed.
The `.png` files in `figures/output/` are preview-only; always submit the `.pdf` versions.